In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [3]:
import my_data_manager as mdm

raw_cats = mdm.load_data("SPLUS_raw_halos")
cats = mdm.scrub_data(raw_cats)
clusters, groups = mdm.separate_clusters(cats)

In [ ]:
import clustering_methods as clustering
import numpy as np

algorithms = {}


algorithms['GMM'] = clustering.run_GMM
algorithms['DBSCAN'] = clustering.run_DBSCAN
algorithms['HDBSCAN'] = clustering.run_HDBSCAN
algorithms['Optics'] = clustering.run_OPTICS
algorithms['Kmeans'] = clustering.run_Kmeans
algorithms['Aglomerative'] = clustering.run_Aglomerative_Clustering
algorithms['Affinity'] = clustering.run_Affinity_Propagation

params = {
    "max_clusters" : 60,
    "covariance_type" : 'full',
    "min_cluster_size" : 4,
    "random_state" : 0,
    "min_eps" : 0.5,
    "max_eps" : np.inf,
    "linkage" : 'ward',
    "clustering_threshold" : 0.5,
    "key_point" : 'location',
    "max_iter" : 1000
}

In [5]:
import numpy as np
import time

def ml_worker(alg):

    predictions = []
    total_time = 0

    for sample in clusters:
    
        X_data = sample.iloc[:,3].values
        Y_data = sample.iloc[:,4].values
        
        data = np.column_stack((X_data, Y_data))
        
        start_time = time.perf_counter()
        labels, probs, clusteres = algorithms[alg](data, params)
        end_time = time.perf_counter()
        delta_time = end_time - start_time
        total_time += delta_time

        predictions.append(labels)
    
    return total_time, predictions

In [13]:
from ds_plus import milaDS
import astro_utils as au
import numpy as np
import time

def dsp_worker():

    predictions = []
    total_time = 0

    for sample in clusters:
        
        X_data = sample["ra"].values
        Y_data = sample["dec"].values
        Z_data = sample["z_app"].values
        
        x_center = (min(X_data) + max(X_data))/2
        y_center = (min(Y_data) + max(Y_data))/2

        X_Kpc, Y_Kpc = au.gal_Mpc_coords(X_data, Y_data, Z_data, x_center, y_center)
        X_Kpc *= 1000
        Y_Kpc *= 1000
        
        Z_clus = np.mean(Z_data)
        V_data = au.los_vel(Z_data, Z_clus)

        start_time = time.perf_counter()
        galaxy_info, grouping, summary = milaDS.DSp_groups(X_data, Y_data, V_data, Z_clus)
        end_time = time.perf_counter()
        delta_time = end_time - start_time
        total_time += delta_time
        
        labels = np.array([row[8] for row in grouping]) #9th column corresponds to the group

        predictions.append(labels)

    return delta_time, predictions



In [14]:
from calsagos import lagasu
from calsagos import utils
from calsagos import clumberi
from astropy.stats import biweight_location
import numpy as np
import time

def calsagos_worker():
    #- S-PLUS mock cosmology
    H_mock = 67.3
    Omega_L_mock = 0.685
    Omega_m_mock = 0.315

    #-- GENERAL PARAMETERS
    range_cuts = 15 # -- number of Gaussians to be fitted in the CLUMBERI and LAGASU implementation
    n_galaxies = 3 # -- number of minimum of galaxies that a group or substructure must have

    predictions = []
    total_time = 0

    for sample in clusters:

        X_data = sample["ra"].values
        Y_data = sample["dec"].values
        Z_data = sample["z_app"].values

        x_cluster = biweight_location(X_data)
        y_cluster = biweight_location(Y_data)
        Z_clus = biweight_location(Z_data)
        
        cluster_mass = sample["log(m_200)"].values[0]
        
        
        start_time = time.perf_counter()
        id_galaxy = indexes = np.arange(len(sample) + 1)
        
        #-- defining the cluster radius
        r200_kpc = utils.calc_radius_finn(cluster_mass, Z_clus, H_mock, Omega_L_mock, Omega_m_mock, "kiloparsec")

        #-- converting the radius in kpc to a radius in angular units
        #-- NOTE: if the user has an estimate of the r200 of the cluster it is not necessary to calculate this quantity
        r200_degree = utils.convert_kpc_to_angular_distance(r200_kpc, Z_clus, H_mock, Omega_m_mock, "degrees") 

        #-- select cluster members
        cluster_members = clumberi.clumberi(id_galaxy, X_data, Y_data, Z_data, Z_clus, x_cluster, y_cluster, range_cuts)

        # -- defining output parameters from clumberi
        id_member = cluster_members[0]
        ra_member = cluster_members[1]
        dec_member = cluster_members[2]
        redshift_member = cluster_members[3]

        #-- estimating the galaxy separation of galaxies in the cluster sample to be used as input in lagasu
        knn_distance = utils.calc_knn_galaxy_distance(ra_member, dec_member, n_galaxies)

        #-- determining the distance to the k-nearest neighbor of each galaxy in the cluster
        knn_galaxy_distance = knn_distance[0]

        typical_separation = utils.best_eps_dbscan(id_member, knn_galaxy_distance)

        #-- Assign galaxies to each substructures
        label_candidates = lagasu.lagasu(id_galaxy, X_data, Y_data, Z_data, 
                            range_cuts, typical_separation, n_galaxies, 'euclidean', 'dbscan', 
                            x_cluster, y_cluster, Z_clus, 
                            r200_degree, 'zspec')
        
        end_time = time.perf_counter()
        delta_time = end_time - start_time
        total_time += delta_time

        #-- defining output parameters from lagasu
        id_candidates = label_candidates[0]
        ra_candidates = label_candidates[1]
        dec_candidates = label_candidates[2]
        redshift_candidates = label_candidates[3]
        label_zcut = label_candidates[4]
        label_final = label_candidates[5]    
    
        predictions.append(label_final)

    return total_time, predictions

In [6]:
from concurrent.futures import ProcessPoolExecutor, as_completed

def run_ml_worker(worker, alg, thread_count, iterations):
    results = []
    
    with ProcessPoolExecutor(max_workers=thread_count) as executor:
            futures = [executor.submit(worker, alg) for _ in range(iterations)]
            for future in as_completed(futures):
                results.append(future.result())
    
    return results

In [11]:
from concurrent.futures import ProcessPoolExecutor, as_completed

def run_worker(worker, thread_count, iterations):
    results = []

    with ProcessPoolExecutor(max_workers=thread_count) as executor:
        futures = [executor.submit(worker) for _ in range(iterations)]

        for future in as_completed(futures):
            results.append(future.result())
    
    return results

In [7]:
import openpyxl
from openpyxl import Workbook

def save_xlsx(results, title):
    wb = Workbook()

    sheet1 = wb.active
    sheet1.title = "Execution Times"
    sheet1.append(["Execution Time (s)"])

    for duration, _ in results:
        sheet1.append([round(duration, 3)])

    for idx, (duration, data) in enumerate(results):
        sheet = wb.create_sheet(title=f"Result_{idx + 1}")

        for row in zip(*data):
            sheet.append(row)

    # Save Excel file
    wb.save(f"{title}.xlsx")

In [ ]:
for alg in algorithms.keys():
    results = run_ml_worker(ml_worker, alg, 4, 10)
    print(alg, ": ", results[0][0])
    save_xlsx(results, ("raw_", alg)) 

GMM :  356.09934435705145
DBSCAN :  1.7375976540024567


/mnt/c/Users/dsolj/OneDrive/Documentos/GitHub/Clustering/.venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/mnt/c/Users/dsolj/OneDrive/Documentos/GitHub/Clustering/.venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/mnt/c/Users/dsolj/OneDrive/Documentos/GitHub/Clustering/.venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/mnt/c/Users/dsolj/OneDrive/Documentos/GitHub/Clustering/.venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/mnt/c/Users

HDBSCAN :  7.811367008987872
Optics :  141.19627427495652
Kmeans :  84.1774776270322
Aglomerative :  102.79077933702865


/mnt/c/Users/dsolj/OneDrive/Documentos/GitHub/Clustering/.venv/lib/python3.11/site-packages/sklearn/cluster/_affinity_propagation.py:140: ConvergenceWarning: Affinity propagation did not converge, this model may return degenerate cluster centers and labels.
  warnings.warn(
/mnt/c/Users/dsolj/OneDrive/Documentos/GitHub/Clustering/.venv/lib/python3.11/site-packages/sklearn/cluster/_affinity_propagation.py:140: ConvergenceWarning: Affinity propagation did not converge, this model may return degenerate cluster centers and labels.
  warnings.warn(
/mnt/c/Users/dsolj/OneDrive/Documentos/GitHub/Clustering/.venv/lib/python3.11/site-packages/sklearn/cluster/_affinity_propagation.py:140: ConvergenceWarning: Affinity propagation did not converge, this model may return degenerate cluster centers and labels.
  warnings.warn(
/mnt/c/Users/dsolj/OneDrive/Documentos/GitHub/Clustering/.venv/lib/python3.11/site-packages/sklearn/cluster/_affinity_propagation.py:140: ConvergenceWarning: Affinity propagat

Affinity :  502.2954548579946


In [ ]:
results = run_worker(calsagos_worker,4,10)
save_xlsx(results, "raw_calsagos")

..starting CLUMBERI....starting CLUMBERI....starting CLUMBERI....starting CLUMBERI..



cluster members: 2409
sigma members: 24
new_cluster_redshift: 0.2876044227723845
velocity dispersion = 1416.6490625394943 [km/s]
starting boostrap estimation of uncertainty on velocity dispersion
cluster members: 2409
sigma members: 24
new_cluster_redshift: 0.2876044227723845
velocity dispersion = 1416.6490625394943 [km/s]
starting boostrap estimation of uncertainty on velocity dispersion
delta_sigma =  19.173261314502838
delta_sigma =  19.173261314502838
cluster members: 2409
sigma members: 24
new_cluster_redshift: 0.2876044227723845
velocity dispersion = 1416.6490625394943 [km/s]
starting boostrap estimation of uncertainty on velocity dispersion
cluster members: 2409
sigma members: 24
new_cluster_redshift: 0.2876044227723845
velocity dispersion = 1416.6490625394943 [km/s]
starting boostrap estimation of uncertainty on velocity dispersion
delta_sigma =  19.173261314502838
delta_sigma =  19.17326131

In [ ]:
results = run_worker(dsp_worker,4,10)
save_xlsx(results,"raw_dsp")